```
# Lab type:  extend
# Course:    NL301 Natural Language Processing with Python
# Lesson:    11 — End-to-End NLP Pipelines
# Task:      Extend a baseline sentiment classifier with four improvements.
```

This lab starts from a working baseline sentiment classifier and asks you to extend it in four progressively more advanced ways: bigrams, threshold calibration, serialisation, and an LLM zero-shot baseline.

## Setup

In [ ]:
!pip install scikit-learn joblib anthropic --quiet
import numpy as np
import joblib
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_recall_curve


## Baseline: complete sentiment classifier

Run this cell — it trains the baseline pipeline that all extensions build on.

In [ ]:
np.random.seed(42)

# Synthetic sentiment dataset
positive = [
    "excellent product highly recommend", "great value for money", "works perfectly",
    "very happy with purchase", "outstanding quality", "love this product",
    "fast delivery good packaging", "exactly as described fantastic",
    "brilliant would buy again", "superb craftsmanship exceeded expectations",
]
negative = [
    "terrible quality broke immediately", "waste of money do not buy",
    "arrived damaged very disappointed", "not as described poor quality",
    "completely useless returned immediately", "worst purchase ever made",
    "misleading description nothing works", "fell apart after one day",
    "customer service unhelpful product defective", "absolutely awful regret buying",
]
texts  = positive + negative
labels = [1]*10 + [0]*10

class TextPreprocessor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        return [t.lower().strip() for t in X]

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.3, random_state=42)

baseline_pipe = Pipeline([
    ('prep',  TextPreprocessor()),
    ('tfidf', TfidfVectorizer(ngram_range=(1, 1))),
    ('clf',   LogisticRegression()),
])
baseline_pipe.fit(X_train, y_train)
baseline_f1 = f1_score(y_test, baseline_pipe.predict(X_test))
print(f"Baseline F1: {baseline_f1:.3f}")


---
## Extension 1: Add bigram support

Change `ngram_range=(1,1)` to `(1,2)`. Bigrams capture phrases like "not good" and "very bad" that unigrams miss.

In [ ]:
# TODO: create bigram_pipe — same as baseline_pipe but with ngram_range=(1,2)
bigram_pipe = None  # replace with your code

# Measure F1 delta
# bigram_f1 = f1_score(y_test, bigram_pipe.predict(X_test))
# print(f"Bigram F1:   {bigram_f1:.3f}  (delta: {bigram_f1 - baseline_f1:+.3f})")


> **Question:** Why do bigrams help for sentiment? Give two examples of bigrams that carry sentiment signal not captured by their individual unigrams.

*(Write your answer here.)*

---
## Extension 2: Threshold calibration

The default `predict()` threshold is 0.5. Use `precision_recall_curve` to find the threshold that maximises F1 on the test set. Report precision and recall at both thresholds.

In [ ]:
# TODO:
# 1. Get predicted probabilities: baseline_pipe.predict_proba(X_test)[:, 1]
# 2. Use precision_recall_curve(y_test, proba)
# 3. Compute F1 at each threshold: 2*P*R / (P+R+1e-9)
# 4. Find best_threshold = thresholds[np.argmax(f1_scores)]
# 5. Print: threshold, precision, recall, F1 at 0.5 vs optimal

proba = None  # replace with your code
best_threshold = 0.5  # replace with computed threshold


> **Question:** In a spam filter, would you prefer higher precision or higher recall? In a medical diagnosis system flagging cancer? Explain the trade-off.

*(Write your answer here.)*

---
## Extension 3: Serialise the full pipeline

Save the pipeline + threshold to disk using `joblib`. Write a `predict_sentiment(text)` function that loads from disk and applies the threshold.

In [ ]:
MODEL_PATH = '/tmp/sentiment_pipeline.joblib'

# TODO:
# 1. Retrain the best pipeline (bigram or baseline) on full training data
# 2. Compute best_threshold using precision_recall_curve on held-out set
# 3. joblib.dump({'pipeline': pipe, 'threshold': best_threshold}, MODEL_PATH)
# 4. Write predict_sentiment(text: str) -> dict that:
#    - Loads the model from MODEL_PATH
#    - Returns {'label': 'positive'/'negative', 'confidence': float}

def predict_sentiment(text: str) -> dict:
    pass  # replace with your code

# Test on three examples
test_examples = [
    "absolutely brilliant product, exceeded all my expectations",
    "not great, arrived damaged and support was unhelpful",
    "it works fine, nothing special",
]
for t in test_examples:
    result = predict_sentiment(t)
    if result:
        print(f"  {result['label']:10s} ({result['confidence']:.3f})  {t}")


> **Question:** Why use `joblib` rather than `pickle` for scikit-learn objects? What else would you need to serialise for a production system that preprocesses text with a custom tokeniser?

*(Write your answer here.)*

---
## Extension 4: LLM zero-shot baseline

Add an LLM classifier using the Anthropic API and compare its F1 to the trained pipeline on 20 test examples. Requires `ANTHROPIC_API_KEY`.

In [ ]:
import anthropic

client = anthropic.Anthropic()

def classify_zero_shot(text: str) -> int:
    """Return 1 (positive) or 0 (negative) using Claude zero-shot."""
    # TODO: write a prompt that returns exactly 'positive' or 'negative'
    # then map to 1 / 0
    pass

# Compare on 20 test examples (use all 20 from X_train + X_test)
test_20  = texts[:20]
labels_20 = labels[:20]

# Uncomment when ready:
# llm_preds = [classify_zero_shot(t) for t in test_20]
# pipe_preds = baseline_pipe.predict(test_20)
# print(f"Trained pipeline F1: {f1_score(labels_20, pipe_preds):.3f}")
# print(f"LLM zero-shot F1:    {f1_score(labels_20, llm_preds):.3f}")


> **Question:** Under what conditions would you choose the trained sklearn pipeline over LLM zero-shot? Consider: latency, cost, data availability, domain specificity, and the volume of predictions per day.

*(Write your answer here.)*

---
## Summary

Answer all four questions:

1. Bigrams: ___
2. Threshold calibration: ___
3. Serialisation: ___
4. LLM vs trained pipeline: ___